# Scientific Ablation Study & Deep Manifold Analysis
**Objective**: Rigorously evaluate ECG reconstruction baselines (Mason-CNN, cNVAE) against the proposed **Neural Inverse Wavelet Transform (NIWT)**.

## key Hypotheses Evaluation
This notebook implements the "Multi-Level Evaluation Framework" proposed in the paper:
1.  **Individual-Level Fidelity**: Testing the Presacan et al. (2025) finding that GANs learn population averages (Regression-to-Mean). We use Ammonia Plots (Error vs. True Amplitude).
2.  **Spectral Fidelity**: Does NIWT's Ricker basis better preserve High-Frequency QRS components compared to CNN smoothing?
3.  **Lead Independence**: Verifying if Parallel Decoders successfully disentangle V1 (RV) and V6 (LV) morphology.
4.  **Latent Topology**: Validating that the Ricker-parameter space learns physiological clustering better than dense latent spaces.

## 1. Setup & Model Loading
Loading Mason, cNVAE, and their Ricker (NIWT) variants.

In [ ]:
import torch
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Advanced Manifold Imports
try: import umap
except ImportError: umap = None

# Path Configuration
sys.path.append("/home/mithunmanivannan")
sys.path.append("/home/mithunmanivannan/third_party/cNVAE_ECG/conditional")
sys.path.append("/home/mithunmanivannan/third_party/ecg_reconstruction")
os.chdir("/home/mithunmanivannan")

# Model Imports
from src.reconstruction.learn_functions.wrappers import MasonWrapper, CNVAEReconstructor, BNVAEArgs
from src.reconstruction.learn_functions.mason_ricker import MasonRicker
from src.reconstruction.learn_functions.cnvae_ricker import CNVAERicker
from src.data.multi_source_dataset import MultiSourceECGDataset

def load_checkpoint(model_name, phase):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    path = f"checkpoints/ablations/{model_name}_{phase}.pt"
    print(f"Attempting to load: {path}")
    if not os.path.exists(path): 
        print(f"!! Checkpoint not found: {path} !!")
        return None
    
    if model_name == 'mason':
        model = MasonWrapper(device).to(device)
    elif model_name == 'mason_ricker':
        # NIWT: Mason Encoder + Ricker Decoder
        model = MasonRicker(in_leads=3, out_leads=12, seq_len=5000, 
                            num_wavelets=64, use_residual=True).to(device)
    elif model_name == 'cnvae':
        args = BNVAEArgs()
        model = CNVAEReconstructor(args, device, num_mixtures=1).to(device)
    elif model_name == 'cnvae_ricker':
        args = BNVAEArgs()
        # Assuming 3-lead input based on project context
        args.num_input_channels = 3
        model = CNVAERicker(args, num_wavelets=64).to(device)
    else:
        print(f"Unknown model: {model_name}")
        return None
        
    try: 
        # Handle DataParallel state dicts if necessary
        state_dict = torch.load(path, map_location=device)
        # Clean state dict keys if needed
        # state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict)
        model.eval()
        return model
    except Exception as e:
        print(f"Error loading state dict: {e}")
        return None

# Load Validation Data
print("Loading Validation Dataset...")
sources = [{"name": "PTB-XL", "path": "data/ptbxl_tensors", "format": "pt"}]
val_ds = MultiSourceECGDataset(split='val', sources=sources, target_len=5000, normalization='min_max')
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=64, shuffle=True)

# Get a fixed batch for consistent comparison
fixed_batch = next(iter(val_loader))
X_val = fixed_batch['input']
Y_val = fixed_batch['target']
if 'label' in fixed_batch: 
    Labels_val = fixed_batch['label']
    Labels_idx = np.argmax(Labels_val.numpy(), axis=1)
else:
    Labels_val = None
    Labels_idx = None

## 2. Spectral Coherence Analysis (Frequency Fidelity)
**Hypothesis**: Convolutional models (Mason) act as low-pass filters, smoothing out sharp QRS details. NIWT (Ricker) should preserve high-frequency content due to wavelet support.

In [ ]:
def run_spectral_analysis(models_dict, x, y_true):
    # Robust device check: Handle both lists and generators for parameters()
    first_model = next(iter(models_dict.values()))
    device = next(iter(first_model.parameters())).device
    x = x.to(device)
    
    psd_results = {}
    
    # Calculate True PSD
    # Flatten leads: [B*12, L]
    # Use reshape to handle potential non-contiguous memory from slicing/permuting
    y_flat = y_true.reshape(-1, 5000).numpy()
    freqs, psd_true = signal.welch(y_flat, fs=500, nperseg=1024)
    psd_results['Ground Truth'] = np.mean(psd_true, axis=0)
    
    plt.figure(figsize=(10, 6))
    plt.plot(freqs, 10 * np.log10(psd_results['Ground Truth']), 'k-', linewidth=2, label='Ground Truth')
    
    for name, model in models_dict.items():
        if model is None: continue
        with torch.no_grad():
            out = model(x)
            if isinstance(out, tuple): recon = out[0]
            else: recon = out
        
        r_flat = recon.cpu().reshape(-1, 5000).numpy()
        _, psd_pred = signal.welch(r_flat, fs=500, nperseg=1024)
        
        plt.plot(freqs, 10 * np.log10(np.mean(psd_pred, axis=0)), label=f"{name}")
        
    plt.title("Spectral Fidelity: Power Spectral Density (Log Scale)")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("PSD (dB/Hz)")
    plt.xlim(0, 100) # ECG dominant frequencies
    plt.legend()
    plt.grid(True, which='both', linestyle='--', alpha=0.7)
    
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/spectral-fidelity.png', dpi=300)
    plt.show()

# Load Models for Analysis
active_models = {}
for m in ['mason', 'mason_ricker', 'cnvae']:
    loaded = load_checkpoint(m, 'phase2')
    if loaded: active_models[m] = loaded

if active_models:
    run_spectral_analysis(active_models, X_val, Y_val)

## 3. Individual-Level Fidelity (Presacan Validation)
**Hypothesis**: Baseline GANs/MSE models exhibit "Regression to the Mean", failing to capture extreme amplitudes. 
**Method**: Plot Reconstruction Error (y-axis) vs. True Amplitude (x-axis). A high correlation ($R^2 > 0.5$) indicates bias (scaling failure).

In [ ]:
def run_presacan_check(models_dict, x, y_true):
    # Analyzing V3 (Lead Index 8)
    lead_idx = 8 
    lead_name = "V3"
    
    first_model = next(iter(models_dict.values()))
    device = next(iter(first_model.parameters())).device
    x = x.to(device)
    
    y_t_lead = y_true[:, lead_idx, :].reshape(-1).numpy()
    
    # Subsample for plotting
    idx = np.random.choice(len(y_t_lead), 5000, replace=False)
    y_sample = y_t_lead[idx]
    
    fig, axes = plt.subplots(1, len(models_dict), figsize=(6*len(models_dict), 5), sharey=True)
    if len(models_dict) == 1: axes = [axes]
    
    results = []

    for i, (name, model) in enumerate(models_dict.items()):
        if model is None: continue
        with torch.no_grad():
            out = model(x)
            if isinstance(out, tuple): recon = out[0]
            else: recon = out
        
        r_lead = recon[:, lead_idx, :].cpu().reshape(-1).numpy()
        error = np.abs(r_lead - y_t_lead)
        
        # Correlation
        r_val = np.corrcoef(np.abs(y_t_lead), error)[0,1]
        results.append({'Model': name, 'R_Error_Amp': r_val})
        
        # Plot
        ax = axes[i]
        ax.scatter(y_sample, error[idx], alpha=0.1, s=1)
        # Trendline
        m, b = np.polyfit(y_sample, error[idx], 1)
        ax.plot(y_sample, m*y_sample + b, 'r--', lw=1)
        
        ax.set_title(f"{name}\nError Scaling (R={r_val:.3f})")
        ax.set_xlabel(f"True Amplitude ({lead_name})")
        if i==0: ax.set_ylabel("Reconstruction Error (Abs)")
        
    plt.suptitle("Individual Fidelity: Does high amplitude cause high error? (Ammonia Plot)")
    plt.tight_layout()
    
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/ammonia-plots.png', dpi=300)
    plt.show()
    
    return pd.DataFrame(results)

if active_models:    
    fidelity_df = run_presacan_check(active_models, X_val, Y_val)
    display(fidelity_df)

## 4. Lead Independence (Parallel Decoder Check)
**Hypothesis**: Shared-network models (Mason) create correlated errors across leads (smearing). NIWT's Parallel Ricker Decoders should decouple leads, allowing V1 and V6 to have independent morphology errors.

In [ ]:
def run_lead_independence(models_dict, x, y_true):
    first_model = next(iter(models_dict.values()))
    device = next(iter(first_model.parameters())).device
    x = x.to(device)
    
    for name, model in models_dict.items():
        with torch.no_grad():
            out = model(x)
            if isinstance(out, tuple): recon = out[0]
            else: recon = out
            
        # Calculate per-lead MSE
        # [B, 12, L] -> Mean over B, L -> [12]
        mse_per_lead = torch.mean((recon - y_true.to(recon.device))**2, dim=(0,2)).cpu().numpy()
        
        # Calculate Error Correlation Matrix between leads
        # Does an error in V1 imply an error in V6?
        # Residuals: [B, 12, L]
        res = (recon - y_true.to(recon.device)).cpu().numpy()
        # Reshape to [B*L, 12] to correlate leads over all time/samples
        res_flat = np.transpose(res, (0, 2, 1)).reshape(-1, 12)
        corr_matrix = np.corrcoef(res_flat, rowvar=False)
        
        # Plot
        leads = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr_matrix, xticklabels=leads, yticklabels=leads, cmap='vlag', vmin=-1, vmax=1, annot=False)
        plt.title(f"{name}: Error Correlation Matrix")
        
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/error-corr-{name}.png', dpi=300)
        plt.show()

if active_models:
    run_lead_independence(active_models, X_val, Y_val)

## 5. Latent Manifold Topology
Visualizing the internal representation of the models. For NIWT, we look at the **Ricker Parameters** (Alpha/Mu/Sigma) as the latent space.

In [ ]:
def extract_latent(model, x):
    device = next(iter(model.parameters())).device
    x = x.to(device)
    with torch.no_grad():
        if isinstance(model, MasonWrapper):
            features = []
            def hook(module, inp, out):
                if isinstance(out, list): t = torch.stack(out, dim=1)
                else: t = out
                if t.ndim > 2: t = t.mean(dim=2)
                features.append(t)
            
            try:
                h = model.model.middle_network.register_forward_hook(hook)
                model(x)
                h.remove()
                return features[0].cpu().numpy()
            except:
                return x.mean(dim=2).cpu().numpy()
                
        elif isinstance(model, MasonRicker):
            x_list = [x[:, i : i + 1, :] for i in range(model.in_leads)]
            f = model.input_network(x_list)
            f_cat = torch.cat(f, dim=1)
            mid = model.middle_network(f_cat)
            params = model.projector(mid)
            return params.cpu().numpy()
            
        elif isinstance(model, CNVAEReconstructor):
            # Robust Hook
            feats = []
            def hook(m, i, o):
                # Handle potential non-3D outputs robustly
                if isinstance(o, torch.Tensor):
                    if o.ndim == 3:
                        feats.append(o.mean(dim=2))
                    elif o.ndim == 2: 
                        feats.append(o) # Already pooled?
                    else:
                        # Handle 4D or others by flattening
                        feats.append(o.view(o.size(0), -1))
                else:
                    print(f"Warning: Hook captured non-tensor: {type(o)}")

            h = model.model.stem.register_forward_hook(hook)
            model(x)
            h.remove()
            if len(feats) > 0:
                return feats[0].cpu().numpy()
            else:
                print("Warning: No features captured from cNVAE stem")
                return None
            
    return None

def run_manifold(models_dict, x, labels):
    if labels is None: 
        print("No labels for manifold coloring.")
        return
        
    fig, axes = plt.subplots(1, len(models_dict), figsize=(5*len(models_dict), 5))
    if len(models_dict) == 1: axes = [axes]
    
    for i, (name, model) in enumerate(models_dict.items()):
        latents = extract_latent(model, x)
        if latents is None: continue
        
        ax = axes[i]
        
        # UMAP with fallback to t-SNE
        proj = None
        algo_name = "t-SNE"
        
        if umap:
            try:
                proj = umap.UMAP(n_neighbors=15, min_dist=0.1).fit_transform(latents)
                algo_name = "UMAP"
            except Exception as e:
                print(f"UMAP failed: {e}. Falling back to t-SNE.")
                
        if proj is None:
            # Fallback to TSNE (imported from sklearn.manifold)
            try:
                # Reducing dim to 50 via PCA first if high dim to speed up t-SNE
                if latents.shape[1] > 50:
                    latents = PCA(n_components=50).fit_transform(latents)
                
                # REMOVED n_iter for compatibility
                proj = TSNE(n_components=2, perplexity=30).fit_transform(latents)
            except Exception as e:
                print(f"t-SNE failed for {name}: {e}")
                continue

        sns.scatterplot(x=proj[:,0], y=proj[:,1], hue=labels, palette='tab10', s=10, ax=ax, legend=False)
        ax.set_title(f"{name} Latent Space ({algo_name})")
            
    # Save the figure for the report
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/latent-topology.png', dpi=300, bbox_inches='tight')
    print("Saved figure to figures/latent-topology.png")
    plt.show()

if active_models and Labels_idx is not None:
    run_manifold(active_models, X_val, Labels_idx)

In [ ]:
# Phase 3 Verification: Hallucination Analysis
# Measuring the "Leakage Matrix" to confirm structured disentanglement reduces crosstalk.

%run analysis/measure_hallucination.py